In [21]:
import json
import os
import shutil
import re

In [22]:
import json
import os
import shutil

def process_node(node):
    if 'raw_text' in node:
        return node['raw_text'].strip()

    if 'AND' in node:
        left_text = process_node(node['AND']['left'])
        right_text = process_node(node['AND']['right'])
        return combine_texts(left_text, right_text, '[AND]')

    if 'OR' in node:
        left_text = process_node(node['OR']['left'])
        right_text = process_node(node['OR']['right'])
        return combine_texts(left_text, right_text, '[OR]')

    if 'NOT' in node:
        left_text = process_node(node['NOT']['left'])
        return f"[NOT] {left_text}"

    return ""

def combine_texts(left_text, right_text, operator):
    if operator == '[AND]' and (left_text.strip().endswith('.') or right_text.strip().startswith('-')):
        return f"{left_text}\n{right_text}"
    return f"{left_text} {operator} {right_text}"

def json_to_text(json_file_path):
    with open(json_file_path, 'r', encoding="utf-8") as f:
        data = json.load(f)
    text = process_node(data)
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    return '\n'.join(lines)

def process_all_files_in_directory(directory_path, output_directory, failure_directory):
    os.makedirs(output_directory, exist_ok=True)
    os.makedirs(failure_directory, exist_ok=True)

    for filename in os.listdir(directory_path):
        if filename.endswith("_exc.json") or filename.endswith("_inc.json"):
            json_file_path = os.path.join(directory_path, filename)
            try:
                output_text = json_to_text(json_file_path)
                output_file_path = os.path.join(output_directory, os.path.splitext(filename)[0] + ".txt")
                with open(output_file_path, 'w', encoding="utf-8") as output_file:
                    output_file.write(output_text)
                print(f"Processed file: {filename}")
            except Exception as e:
                print(f"Failed to process file: {filename} - Error: {e}")
                shutil.move(json_file_path, os.path.join(failure_directory, filename))
                print(os.path.join(failure_directory, filename))

### Json Output zu Txt Files mit Operatoren

In [23]:
model_output_folder = 'model_output/' #versuch_ordner + 
models = [folder for folder in os.listdir(model_output_folder) if os.path.isdir(os.path.join(model_output_folder, folder))]

model = models[1]
#model = "Llama-3-70B-Instruct_3_shot"#"Llama3_70b_Fine-Tuned_ep10"

In [24]:

model_name = f"model_output/{model}"  # Ersetzen Sie dies durch den tatsächlichen Modellnamen
input_directory_path = os.path.join(model_name, "ready")
output_directory_path = os.path.join(model_name, "processed_output")
failure_directory_path = os.path.join(model_name, "failure")
process_all_files_in_directory(input_directory_path, output_directory_path, failure_directory_path)

Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03923933_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03923946_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03923959_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03923998_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03924024_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03924037_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03924180_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03924206_exc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03924232_exc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03924284_exc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03924284_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03924362_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03924375_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03924453_exc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep10_NCT03924479_inc.

In [25]:
ic = "Inclusion Criteria:"
ec = "Exclusion Criteria:"

In [26]:
def combine_files(input_dir, output_dir):
    def read_file(filepath):
        with open(filepath, 'r', encoding="utf-8") as file:
            return file.read()

    def write_combined_file(nct_number, inc_text, exc_text, output_dir):
        output_filepath = os.path.join(output_dir, f'NCT{nct_number}.txt')
        with open(output_filepath, 'w', encoding="utf-8") as file:
            file.write(f"Inclusion Criteria:\n{inc_text}\n")
            file.write(f"Exclusion Criteria:\n{exc_text}\n")

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    files = os.listdir(input_dir)
    nct_numbers = set()

    pattern = re.compile(r'NCT\d+')

    for file in files:
        match = pattern.search(file)
        if match:
            nct_number = match.group()
            nct_numbers.add(nct_number)
            print(f"Processing {nct_number}")

    for nct_number in nct_numbers:
        inc_files = [f for f in files if nct_number in f and '_inc' in f]
        exc_files = [f for f in files if nct_number in f and '_exc' in f]

        if inc_files and exc_files:
            inc_text = '\n'.join(read_file(os.path.join(input_dir, f)) for f in inc_files)
            exc_text = '\n'.join(read_file(os.path.join(input_dir, f)) for f in exc_files)
            print(inc_text)
            print(exc_text)
            write_combined_file(nct_number, inc_text, exc_text, output_dir)

In [27]:
input_dir = f'model_output/{model}/processed_output'
output_dir = f'model_output/{model}/combined_output'

combine_files(input_dir, output_dir)

Processing NCT03923933
Processing NCT03923946
Processing NCT03923959
Processing NCT03923998
Processing NCT03924024
Processing NCT03924037
Processing NCT03924180
Processing NCT03924206
Processing NCT03924232
Processing NCT03924284
Processing NCT03924284
Processing NCT03924362
Processing NCT03924375
Processing NCT03924453
Processing NCT03924479
Processing NCT03924492
Processing NCT03924492
Processing NCT03924505
Processing NCT03924544
Processing NCT03924609
Processing NCT03924609
Processing NCT03924687
Processing NCT03924726
Processing NCT03924752
Processing NCT03924817
Processing NCT03924908
Processing NCT03924921
Processing NCT03925116
Processing NCT03925142
Processing NCT03925168
Processing NCT03925181
Processing NCT03925298
Processing NCT03925363
Processing NCT03925389
Processing NCT03925441
Processing NCT03925454
Processing NCT03925506
Processing NCT03925584
Processing NCT03925610
Processing NCT03925649
Processing NCT03925662
Processing NCT03925675
Processing NCT03925727
Processing 